# Lab 3 — Getting the time axis right
### EP0418 · Topic 3 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`sg_taxi_hourly.csv` itself. Have the printed Lab 3 sheet beside you for the boxes
you write in.

---
Singapore's taxi feed on `data.gov.sg` reports how many taxis are free across the
island, about once an hour. It records the moment each reading was taken, so the
readings are about an hour apart, not exactly. Today you:

- read the timestamps and see how they drift (Task 1);
- put the readings on a regular hourly grid (Task 2) and count the empty hours (Task 3);
- compare each hour with the same hour last week (Task 4);
- see the daily and weekly cycles (Task 5).

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Prompts** in grey quote boxes start with **Rules:** (for example, do not drop
  rows; show the row count before and after), then **Task:**. Keep the Rules when
  you change a prompt. A prompt with no example is **zero-shot**; one example of the
  output you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

CSV = "sg_taxi_hourly.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows")
raw.head()

## Task 1 — Read the timestamps before anything else (20 minutes)

*Topic 3: the time column is data too. Readings are not always taken exactly on the hour.*

**What each field means.** The file `sg_taxi_hourly.csv`, loaded as `raw` by the cell
above, has 1,344 readings from 29 May to 24 July 2026: eight weeks.

| Where | Field | What it means | Example |
|---|---|---|---|
| `raw` | `timestamp` | the moment the reading was taken, Singapore time (`+08:00`), to the second | 2026-05-29 23:59:48+08:00 |
| `raw` | `taxi_count` | how many taxis were free for hire across Singapore at that moment | 3008 |

Task 1 has two parts: **1A**, read the first six timestamps; **1B**, chart the
readings and the gaps between them.

### 1A · Read the first six timestamps

1. **Run the cell below.** It prints the first six timestamps, then the five most
   common gaps between one reading and the next.
2. **Read the times.** Are they exactly on the hour, like `01:00:00`? Write your
   answer in the box *On the hour?* just under the code cell, before reading on.

In [ ]:
print(raw.timestamp.head(6).to_string(index=False))
print()
print("gaps between consecutive readings:")
print(raw.timestamp.diff().value_counts().head(5).to_string())

**On the hour? (yes / no, and an example time):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

No. The first reading is `23:59:48`, then `00:59:50`, `01:59:53`, `02:59:55`: each
one a few seconds before the hour, and creeping a couple of seconds later every
time. A good answer gives one of those times as the example. The common wrong answer
is "yes", either because the times look close enough or because the sixth reading
happens to be `05:00:00`. That one is on the hour by luck, not by design. The feed
records the moment the reading was taken, so the gaps are about one hour, not
exactly one hour, and that is why the readings must go on a regular grid before you
count back 168 hours to the same hour last week (Task 4).
</details>

**What you should see:** times like `23:59:48` and `00:59:50`, a few seconds off
the hour. The five most common gaps are `01:00:02` (423 times), `01:00:03` (392),
`00:59:33` (141), `00:59:32` (129) and `01:00:33` (111): all about one hour, none
exactly one hour.

### 1B · Chart the readings and the gaps between them

1. **Run the cell below.** It draws two charts: every reading, and the hours since
   the previous reading.
2. **Look for** the daily rhythm in the first chart. In the second, look for
   readings that came early or late.

In [ ]:
show.series(raw, col="taxi_count", title="Taxis available, every reading, eight weeks")
hours_since_previous = raw.timestamp.diff().dt.total_seconds() / 3600
show.series(pd.Series(hours_since_previous.values, index=raw.timestamp),
            title="Hours since the previous reading", ylabel="hours", hline=1)

**What you should see:** in the first chart, a strong daily cycle with a weekly one
on top of it. In the second chart, the line sits on the dashed 1-hour line almost
everywhere. Only 21 readings came more than two minutes early or late, between
0.57 hours (34 minutes) and 1.18 hours (71 minutes) after the one before.

The problem is one that neither chart can show. The readings creep a few seconds
later each hour, and now and then one slips past the hour mark. That clock hour then
has no reading and the next hour has two, yet the gap between the two readings is a
normal hour. You need a regular grid to see those empty hours: Task 2 makes one.

**What neither chart can show**, on six real readings from 30 May (it plays slowly, about a minute):
![Six real readings from 30 May drift across the hour mark: the 04:00 hour has no reading and the 05:00 hour has two, although the gap between them is a normal 1 h 0 min 2 s](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_drift.gif)

### Why it matters: a shift counts rows, not hours

`23:59:48`, then `00:59:50`, then `01:59:53`. The feed returns the moment each
reading was taken, not a tidy clock tick. This is **irregular sampling** (Topic 3).

If you want "the same hour last week" and you write `.shift(168)`, you are
shifting by **168 rows**, not 168 hours. This feed never skipped a reading (every
gap is between 34 and 71 minutes), so here 168 rows back always lands between
6 days 23 h 33 min and 7 days 0 h 10 min back: within half an hour of a week. A
feed that missed even one reading in that week would put every row after it an
hour off, and nothing would warn you.

So put the readings on a regular grid first, and shift after that.

## Task 2 — Put the readings on a regular hourly grid (30 minutes)

*Topic 3, Resampling: you choose the slot size, what goes in each slot, and what an
empty slot shows.*

**Why a grid?** Task 4 compares each hour with the same hour last week, 168 hours
back. On the raw readings you can only count back rows, and Task 1 showed that a row
is not always one clock hour: some hours caught two readings and some caught none. A
regular grid gives every clock hour exactly one slot, so 168 slots back is always one
week. In a real system the readings keep arriving and the grid grows hour by hour. In
this lab all eight weeks come from the saved file, so everyone builds the same grid
and the checks give everyone the same answer.

**The goal:** `s`, a Series with one slot per hour, every hour of the eight weeks,
from 29 May 23:00 to 24 July 22:00. The slot size is one hour; the other two
decisions are yours.

**Step 1. Predict, before you ask the AI.** From Task 1: some clock hours caught two
readings, and some caught none. Write two short answers in the box below:

1. An hour caught two readings. What one number should its slot show?
2. An hour caught no reading. What should its slot show, and why?

**My plan:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2. Ask Gemini for the code.** Open Gemini (the **spark icon** at the bottom
centre of the Colab window) and start a new chat (click **⋮** at the top right of the
Gemini panel, then **New chat**, then **Yes**). The prompt below ends with one
sentence hidden: what an hour with no reading must show. **Before you open it, say
your answer** (it is question 2 of your plan in Step 1).

<details><summary><b>Show the prompt</b> (click, then copy all of it into the chat and press Enter)</summary>

> **Rules:** Do not drop any hours. Show the row count before and after, and how many
> hours are empty. Explain in plain words what the code does.
>
> **Task:** I have a DataFrame `raw` with a `timestamp` column and a `taxi_count`
> column. The timestamps are irregular. Resample onto a regular hourly grid, taking
> the mean within each hour, and call the result `s` (a Series of taxi_count). One
> example: on 30 May the readings at 05:00:00 (2232 taxis) and 05:59:33 (1766 taxis)
> both fall in the 05:00 hour, so that slot is 1999.0. An hour with no reading must
> keep its slot in s, left empty (NaN), not 0 and not copied from the hour before.

**The last sentence was the hidden one.** It says two things. Did your answer say both?

- **keep its slot**: otherwise the empty hours vanish, `s` has fewer slots than
  there are hours, and a shift of 168 slots is no longer one week;
- **left empty, not 0**: an hour with no reading is not an hour with no taxis. Zeros
  would look like real readings and pull every average down.
</details>

**One-shot** (Video 2): the prompt shows one worked example, the 05:00 slot on 30 May,
so Gemini copies how a slot is filled, mistakes included. The example also answers
question 1 of your plan: two readings in one hour give their mean, 1,999.

**Step 3. Run Gemini's code, then the check.**

1. Gemini answers by adding a new code cell to your notebook, with a green edge. At
   the bottom of the chat panel, just above the box where you type, is a button
   **Accept and run**. Click it: this keeps the code and
   runs it.
2. If the code appears only in the chat and no green cell appears: copy the code,
   click **+ Code** at the top of the page, paste it into the new cell and run it.
3. If Gemini shows red and green lines inside the check cell (its first line starts
   `# TODO(you)`), click **Cancel**: check cells are never changed.
4. Colab scrolls to the green cell. Scroll back down to this task and run the check
   cell just below this text. You should see `[PASS] regular_cadence`.

**Step 4. Only if the check fails, or Gemini's green cell shows red error text:** in
the same chat, type one sentence with your numbers, for example *s has 1,305 rows,
but the eight weeks have 1,344 hours. Keep every hour, with taxi_count left as NaN
when it has no reading.* Click **Accept and run**, then run the check cell again.
Give the numbers, not just *it didn't work*: with the numbers, Gemini knows what to fix.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `s`), then run this cell.
s = keep("s")

# ---- verification contract — do not edit ----
check.regular_cadence(s, "1h")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **The report:** `[PASS] regular_cadence(1h)`, then *All 1,343 gaps are exactly 1h.*
- **Gemini's green cell** printed the size of `s`: 1,344 hourly slots, one per hour of the eight weeks. More means the grid is finer than hourly; fewer means the empty hours were dropped, and the check fails.

**What the code had to do**, on six real readings from 30 May, and the three ways it can turn out (it plays slowly, about a minute):
![Resampling six real readings from 30 May into hourly slots: each slot takes the mean of its readings and the empty 04:00 slot stays NaN; then the three outcomes, A right, B dropped, C filled](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_grid.gif)

**Run the cell below** and **look for** the same daily and weekly shape as the first chart in Task 1.

In [ ]:
if filled(s):
    show.series(s, title="On a regular hourly grid: one value per hour")

**What you should see:** the same shape as before. The empty hours are single
hours, too narrow to see across eight weeks; Task 3 counts them for you. If the
line drops straight to 0 in places, Gemini's green cell filled the empty hours with zeros: that
is outcome **C** in the clip, and Task 3 gives the sentence that fixes it.

## Task 3 — Count the empty hours (10 minutes)

Resampling does not invent data. It makes a slot for every hour, so you can count
the hours that had no reading.

1. **Run the cell below.** It prints one line: the empty hours in `s`, out of all
   its slots.
2. **Compare** your line with the *Expected output* box and the table **A / B / C**,
   both just under the code cell.
3. **Write two answers** in the box *Empty hours and slots, and my outcome* under the
   table, or on the sheet:
   1. How many empty hours, out of how many slots? (your line, `empty hours: … of …`)
   2. Which outcome, A, B or C, matches your line? (the table **A / B / C**)

In [ ]:
if filled(s):
    print(f"empty hours: {int(s.isna().sum())} of {len(s):,} ({s.isna().mean():.1%})")

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `empty hours: 39 of 1,344 (2.9%)`

**Read the line** and compare:

| | empty hours | slots in `s` | what it means |
|---|---|---|---|
| **A** | 39 | 1,344 | **Right.** Every hour has a slot; the 39 with no reading are empty. |
| **B** | 0 | 1,305 | **Wrong:** the empty hours were dropped. `regular_cadence` in Task 2 fails. |
| **C** | 0 | 1,344 | **Wrong, and no check catches it:** the empty hours were filled, with 0 or with the hour before. |

**If you got A:** write your two answers in the box below.

**If you got B or C:** ask Gemini to fix it. In the same chat as Task 2, type one
sentence with your numbers, for example:
- for **B**: *s has 1,305 rows, but the eight weeks have 1,344 hours. Keep every
  hour, with taxi_count left as NaN when it has no reading. Keep everything else.*
- for **C**: *s has 1,344 rows but 0 empty hours. The 39 hours with no reading must
  stay NaN, not 0 and not copied from the hour before. Fix that and keep everything
  else.*

Click **Accept and run**, run the Task 2 check cell, then run this task's count cell
again: it should now say 39 of 1,344. Give the numbers, not just *it didn't work*:
with the numbers, Gemini knows what to fix; without them, it guesses.

**Empty hours and slots, and my outcome (A, B or C):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Yours should read 39 of 1,344 (2.9%): outcome A. If yours says
0 of 1,344 (outcome C), Gemini filled the empty hours with something, usually zeros
or the previous value, and the chart in Task 2 dropped to 0 or ran flat in places.
If yours says 0 of 1,305 (outcome B), the empty hours were dropped instead (1,344
minus 39). Both look tidy and both are wrong, because an hour with no reading is not
an hour with no taxis. One more thing worth noticing: each of the 39 empty hours is
followed by an hour that caught two readings. The reading drifted across the hour
mark, so one hour got none and the next got two.
</details>

## Task 4 — Compare each hour with the same hour last week (20 minutes)

**Why compare with last week?** Before you trust any forecast, you need to know how
well the simplest guess does. If a cleverer method cannot beat *the same hour last
week*, it is not worth using.

On the regular grid one slot is one hour, so `shift(168)` moves every value exactly
one week later (7 days × 24 hours). The shifted copy is a guess for every hour: *the
same hour last week*. This guess is called the **seasonal-naive** baseline (Topic 3:
seasonality is a pattern that repeats on a fixed cycle, here every week).

**MAE** (mean absolute error) is the average size of the miss: for each hour,
how far the guess was from the truth, ignoring whether it was too high or too
low, averaged over all hours.

**Watch first**, what the code in this task must do, on two real weeks of `s` (it
plays slowly, about a minute). When it ends, do Step 1.
![shift(168) on the hourly grid: week 1 slides one week later and becomes the guess for week 2; on Wed 10 June at 18:00 there were 3,387 taxis against a guess of 3,052; hours with no value or no guess are skipped; the MAE is about 496 taxis](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_shift.gif)

The steps are tagged with the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

**Step 1.** `PROMPT` **Ask Gemini.** Open Gemini (the **spark icon** at the bottom centre of the Colab window), start a new chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**), paste this and press Enter. To copy it, drag
across the quoted text below from its first word to its last, then press Ctrl+C
(Cmd+C on a Mac):

> **Rules:** Do not change `s`. Say how many rows were used. Explain in plain words
> what the code does.
>
> **Task:** `s` is an hourly pandas Series. Build `snaive`, which is `s` shifted by
> 168 hours. Then compute the mean absolute error between `s` and `snaive`, using only
> rows where both exist, and call it `mae_snaive`.

**Step 2.** `GENERATE` **Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it. If Gemini ever shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**: check cells are never changed.
Colab scrolls to the green cell, even to the very top of the notebook, so after
**Accept and run** scroll back down to this task for Step 3.

**Step 3.** `VERIFY` **Run the check cell below.** You should see the seasonal-naive MAE printed, close to the expected output under the cell, and a NOTE or PASS from `compare_to_baseline`.

**Step 4.** `REFINE` **Only if a check fails, a number looks wrong, or Gemini's green cell
shows red error text:** in the same chat, type one
sentence saying what you saw and what you expected, for example *mae_snaive is 824 but should be about 496. Use only the hours where both s and snaive have values; do not fill the empty ones with 0.* Press Enter,
click **Accept and run** (Gemini may add a new green cell or change its own earlier
one; either is fine), and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `snaive` and `mae_snaive`), then run this cell.
snaive = keep("snaive")
mae_snaive = keep("mae_snaive")

if filled(snaive, mae_snaive):
    print(f"seasonal-naive MAE: {mae_snaive:,.1f}")
    print(f"mean taxi count   : {s.mean():,.0f}")

# ---- verification contract — do not edit ----
# The baseline's MAE is compared with the series' standard deviation, which is
# roughly the error of always guessing the average. The report calls the first
# number "Model" because the check is general; here it is your baseline.
if filled(s):
    check.compare_to_baseline(mae_snaive, s.std(), metric="MAE")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `seasonal-naive MAE: 495.6`
- **Line 2:** `mean taxi count   : 2,375`
- **The report:** `[PASS] compare_to_baseline`. It calls your guess "Model" and compares its MAE with about 720, the spread of `s` (its standard deviation): roughly the miss of always guessing the average.
- **If your MAE is about 824:** the empty values were filled with 0 taxis: the 39 empty hours, and the first week, which has no week before it to copy. **If it is about 411:** hours with no guess were counted as a miss of 0. Use only the hours where both exist.

**Run the cell below.** **Look for** how close "the same hour last week" tracks the
real line, and where it misses.

In [ ]:
if filled(snaive):
    show.compare(s.iloc[-24 * 14:], same_hour_last_week=snaive.iloc[-24 * 14:],
                 title="Last two weeks: actual against the seasonal-naive guess")

**What you should see:** the dashed guess follows the real line closely most of
the time, because the daily cycle repeats. Where they part is where something
unusual happened in one of the two weeks. Those misses are what the MAE averages:
about 496 taxis an hour, against a mean of about 2,375. A better model would have to
explain them.

## Task 5 — See the daily and weekly cycles (10 minutes)

*Topic 3: seasonality, a pattern that repeats every day or every week.*

1. **Run the cell below.** It draws two line charts (dots joined by a line): the
   average by hour of day (24 dots), and by day of the week (7 dots).
2. **Look for** the lowest and highest dots in each chart.

In [ ]:
if filled(s):
    show.profile(s, by="hour", title="Mean taxis available by hour of day")
    show.profile(s, by="dow", title="Mean taxis available by day of week")

**What you should see:** in the first chart, a deep dip in the small hours of the
night and a high, uneven stretch from late morning to late evening. In the second,
two days sit well below the other five. Read the hour of the lowest dot off the
first chart's bottom axis.

3. **Write** the hour at which availability is lowest in the box under the charts.
4. `PROMPT` `GENERATE` `VERIFY` **Optional:** ask Gemini (new chat: **⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**) *Draw a bar chart of the mean of `s` by hour
   of day.* Accept and run its cell. Its lowest bar should sit at the same hour as
   the lowest dot in the first chart above.

**Hour of lowest availability:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

About 3 am. The `03:00` slot has the lowest average, about 1,785 taxis, with 4 am
and 5 am only a little higher (about 1,890 and 1,840). The busiest end is the
evening, about 2,780 at 8 pm. A good answer gives the hour, not a day. The common
slip is reading the second chart, the day-of-week one, and writing "Saturday": that
is the lowest day (about 1,880, with Sunday about 1,940), not the lowest hour.
Another is writing midnight, because the line starts there; at about 2,280 it is far
from the bottom.
</details>

---
## Going further (optional): hourly share prices and their time zone

Real hourly share prices have their own time-axis problems: the market closes at
night and at weekends, and the times can arrive in another time zone.
This uses **yfinance**, a free, unofficial library for Yahoo Finance data, meant for
personal, educational use (pypi.org/project/yfinance).

1. **In the cell below, change** `TRY_LIVE = False` to `TRY_LIVE = True`.
2. `VERIFY` **Run the cell.** It downloads a month of hourly prices for ES3.SI (an ETF of
   Singapore shares) once, saves them, and reuses the saved copy, so the class does
   not hit Yahoo's limits. You should see the time zone and a list of trading days
   with a number of rows each. If Yahoo refuses, try again later. If you see
   *ModuleNotFoundError: yfinance*, click **+ Code**, type `!pip install yfinance` in
   the new cell, run it, then run this cell again.
3. **Answer** questions A, B and C in the box under the cell:

- **A.** How many rows does each trading day have, and which days are missing from
  the list? (the list of dates the cell printed, with a row count for each)
- **B.** What time zone are the timestamps in? (the line `time zone:`)
- **C.** Should you put these prices onto a regular hourly grid, as you did with the
  taxi feed? Answer yes or no, and give one reason. (Nothing printed answers this.
  A hint: an empty hour on the grid would stand for an hour when the market is closed.)

In [ ]:
TRY_LIVE = False      # set to True to try it

if TRY_LIVE:
    import os
    import yfinance as yf          # if this fails: run  !pip install yfinance  first
    if not os.path.exists("es3_hourly.csv"):
        yf.Ticker("ES3.SI").history(period="1mo", interval="1h").to_csv("es3_hourly.csv")
    live = pd.read_csv("es3_hourly.csv", index_col=0)
    live.index = pd.to_datetime(live.index, utc=True).tz_convert("Asia/Singapore")
    print("time zone:", live.index.tz)
    print(live.groupby(live.index.date).size().tail(10))   # rows per trading day

**My answers:**

A.
B.
C.

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Three separate answers.

- A. Every full trading day has the same small number of rows: count them in your own list. The gaps are the nights, the weekends and any public holiday, when the exchange is closed.
- B. Asia/Singapore. The cell set this itself with `tz_convert`: Yahoo sends the times with an offset, and a careless conversion would shift every hour.
- C. No. On a regular hourly grid every night and weekend becomes a run of empty hours, yet nothing is missing: the market was closed. The common wrong answer is "yes, so the shift works". Here the honest step is "one trading hour back", not "one clock hour back".
</details>

---
## Predict: How many taxis at 8 am on Monday? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** Taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. The target is the week of Monday 2026-11-09.

**How it works:** you enter a short function, `predict(history)`, not a number. When
entries close, the league runs your function once on the latest data. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=3

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the average of every Monday 8 o'clock reading so far*, on the lab's own file. The league works the simple rule out again
from the latest data when entries close, so its number will differ.

In [ ]:
fetch("sg_taxi_hourly.csv")
taxi = pd.read_csv("sg_taxi_hourly.csv", parse_dates=["timestamp"])

def predict(history):
    # the baseline: the average of every Monday 8 o'clock reading so far
    ts = history["timestamp"]
    mondays = history[(ts.dt.weekday == 0) & (ts.dt.hour == 8)]
    print("Mondays at 8:", mondays["taxi_count"].tolist())
    return float(mondays["taxi_count"].mean())

print("The baseline predicts", round(predict(taxi), 1), "taxis")

**What you should see:** *The baseline predicts 2399.4 taxis*.

**Step 2.** **Ask Gemini to write your predict.** Start a new
chat (**⋮** › **New chat** › **Yes**), paste the prompt and press Enter.

> **Rules:** Put it in a new cell and do not change the cells above. Write only the function and any import lines it needs. Keep the name *predict* and its one input, *history*, a pandas table with the columns timestamp (one reading per hour, Singapore time) and taxi_count, oldest first. Return one number with float(...). **Task:** Write a predict(history) that estimates this: taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. Do it in a way you think beats the simple rule (the average of every Monday 8 o'clock reading so far). Explain the idea in one sentence.

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(taxi, "taxis")

**What you should see:** *Your predict says* and a number, usually between 1,500 and 4,000, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/taxi_hourly.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: Taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. The target is the week of Monday 2026-11-09 to Sunday 2026-11-15.
The simple rule to beat: the average of every Monday 8 o'clock reading so far.
My idea: WRITE YOUR IDEA HERE

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: timestamp, taxi_count. What they hold: timestamp (one reading per hour, Singapore time) and taxi_count. The timestamp column is timezone-aware (Asia/Singapore): compare it only with timezone-aware times, for example pd.Timestamp("2026-10-26", tz="Asia/Singapore").
3. Do not read any file and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. Return one number with return float(...). Not a list, a table, text, None or NaN.
5. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
6. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code, explain the idea in one sentence.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=3 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: both are normal. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* confirms it. Submit again any time before the close; the latest one counts.

**Write it down** (on the sheet, or double-click here): your function's number (Step 3),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (2399.4 taxis), and an idea that uses the data. The Monday 8 o'clock readings in the file are 3,841, 2,065, 2,460, 2,249, 2,732, 2,040, 2,006 and 1,802. The first is far above the rest: without it the average is about 2,193 instead of 2,399, and the median of all eight is 2,157. Ideas that use this: the median, or the average of the last few Mondays.
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked to put the taxi feed onto a regular hourly grid and “make sure there are no missing values”. It wrote the code below.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Check whether the explanation points out the problem.
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** What does the `fillna(0)` line do to the 39 empty hours, and why does that matter for anything computed from `their_s` afterwards?

**Question 2.** Predict what the two printed lines will say about the minimum and the mean of `their_s`, before and after `fillna(0)`. At step 3 you run it and compare.

<details><summary>Hint 1: where to look</summary>

What value does `fillna(0)` write into an empty hour? Is zero a measurement of taxis, or the absence of one?

</details>
<details><summary>Hint 2: closer to the answer</summary>

The two printed lines show it. The 39 empty hours become hours with 0 taxis: the minimum falls to 0 and the mean drops, and the `regular_cadence` check from Task 2 would still pass.

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the *after* line no longer shows min 0; it matches the *before* line (min 24, mean about 2,375).

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_s = raw.set_index("timestamp").taxi_count.resample("h").mean()
# print("before: min", their_s.min(), " mean", round(their_s.mean()))
# their_s = their_s.fillna(0)
# print("after:  min", their_s.min(), " mean", round(their_s.mean()))

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: why the taxi timestamps are not on the hour; what resampling onto a regular grid does to the gaps; why shift(168) only means one week on a regular grid; what a seasonal-naive forecast is; what mean absolute error measures. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

**Optional: the Trading League opens today.** `shift()` on a regular grid is
exactly what a trading rule is built from: yesterday's close, last week's close, a
moving average. If you want a game to play with it, open the League notebook from
the scoreboard page (https://stanleysp-eng.github.io/EP0418-labs/), do Tasks 1 to
3, and your entry is on the board. Not graded, not required, open until the end of
Week 7.

**One thing to notice:** the smallest reading in this file is 24 taxis across all of
Singapore, at 21:54 on 17 July, when the average is about 2,375. That is not
plausible: it is a real glitch in the feed, not something we added. Lab 4 is about
what to do with values like that.

**Before you leave**, fill in the *Mistake caught* box below (double-click it), or
write it on the last page of the Lab 3 sheet: which outcome, **A**, **B** or **C**
(Task 3), did Gemini's green cell for Task 2 give on the first try? If it gave A first time, use the
read-before-you-run code instead, for example *I expected empty hours to stay empty ·
they became 0 taxis · the minimum fell from 24 to 0.*

**Mistake caught:** *(what I expected · what was there · how I found it. If Gemini gave outcome A first time, use the read-before-you-run example in the text just above this box.)*

*(Your answer box: double-click this cell. Type your answer here.)*